# 04: Diagnostics

Additional analyses outside the paper's stated design. All outputs go to `results/diagnostics/`. All settings are reported; none is selected.

1. **Image–text interaction features** (truth probe): CLIP image–text cosine as a single feature, and the element-wise product of the image and text embeddings.
2. **Bag-of-words baseline** (relation identity): `CountVectorizer` + logistic regression on raw captions.
3. **Regularization sweep and scaling** (truth probe): `C` in `[0.01, 0.1, 1, 10, 100, 1000]`, with and without `StandardScaler`.

Unless stated otherwise: `LogisticRegression(max_iter=1000, solver="lbfgs", random_state=42)`, 5 folds, `random_state=42`, `min_samples=50`.

## 1. Setup

In [1]:
import sys, os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from datasets import load_dataset
from sklearn.exceptions import ConvergenceWarning

PROJECT_ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()).resolve()
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

CACHE_DIR = Path(os.getenv("CACHE_DIR", "results/embeddings"))
if not CACHE_DIR.is_absolute():
    CACHE_DIR = PROJECT_ROOT / CACHE_DIR
RESULTS_DIR = PROJECT_ROOT / "results"
OUT_DIR = RESULTS_DIR / "diagnostics"
OUT_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(42)
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 200)

from src.probing import probe_truth_by_relation

MODELS = ["sbert", "clip_text", "clip_image", "clip_concat"]
embeddings = {m: np.load(CACHE_DIR / f"{m}_vsr_train.npy") for m in MODELS}

train_data = load_dataset("cambridgeltl/vsr_random")["train"]
relation_types = np.array(train_data["relation"])
binary_labels = np.array([int(v) for v in train_data["label"]])
captions = np.array(train_data["caption"])
image_files = np.array(train_data["image"])

GROUPS = {
    "sbert": captions,
    "clip_text": captions,
    "clip_image": image_files,
    "clip_concat": [captions, image_files],
}
TOL = 1e-9
print({m: e.shape for m, e in embeddings.items()})

{'sbert': (7680, 768), 'clip_text': (7680, 512), 'clip_image': (7680, 512), 'clip_concat': (7680, 1024)}


In [2]:
def summarize_truth(df: pd.DataFrame) -> dict:
    """Across-relation summary of one probe_truth_by_relation result."""
    f1 = df["f1_macro_mean"]
    return {
        "mean_f1": f1.mean(),
        "std across relations": f1.std(ddof=0),
        "median_f1": f1.median(),
        "min_f1": f1.min(),
        "max_f1": f1.max(),
        "mean fold std": df["f1_macro_std"].mean(),
        "baseline_most_frequent_mean_f1": df["baseline_most_frequent_f1_macro"].mean(),
        "baseline_stratified_mean_f1": df["baseline_stratified_f1_macro"].mean(),
        "n_above_stratified_baseline": int((f1 - df["baseline_stratified_f1_macro"] >= TOL).sum()),
        "n_tied_most_frequent_baseline": int(((f1 - df["baseline_most_frequent_f1_macro"]).abs() < TOL).sum()),
        "n_relations": len(df),
    }

def run_truth(X, groups=None, **kwargs):
    """Run the truth probe and count lbfgs convergence warnings."""
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        df = probe_truth_by_relation(X, relation_types, binary_labels, n_splits=5, min_samples=50,
                                     groups=groups, **kwargs)
    n_conv = sum(issubclass(w.category, ConvergenceWarning) for w in caught)
    return df, n_conv

## 2. Image–Text Interaction Features (truth probe)

`clip_concat_vsr_train.npy` holds the projected image embedding in columns 0–511 and the projected text embedding in columns 512–1023 (shared CLIP space). Each half is re-normalized to unit length, giving `u` (image) and `v` (text).

- `cosine`: one feature, `u · v`
- `product`: 512 features, `u * v` element-wise

Each is run at `C=1.0`, with and without `StandardScaler` (fit on the training fold), under stratified folds and under joint caption+image grouped folds. The `concat` rows are the unmodified CLIP Concat condition for reference.

In [3]:
concat = embeddings["clip_concat"]
u = concat[:, :512] / np.linalg.norm(concat[:, :512], axis=1, keepdims=True)
v = concat[:, 512:] / np.linalg.norm(concat[:, 512:], axis=1, keepdims=True)

features = {
    "cosine": (u * v).sum(axis=1, keepdims=True),
    "product": u * v,
    "concat": concat,
}
print("cosine feature: mean %.4f, std %.4f, min %.4f, max %.4f" % (
    features["cosine"].mean(), features["cosine"].std(), features["cosine"].min(), features["cosine"].max()))
print("product feature: per-dimension std, mean over dims %.5f" % features["product"].std(axis=0).mean())

detail, rows = [], []
for name, X in features.items():
    for standardize in [False, True]:
        for cv_name, groups in [("stratified", None), ("grouped (caption+image joint)", GROUPS["clip_concat"])]:
            df, n_conv = run_truth(X, groups=groups, C=1.0, standardize=standardize)
            df.insert(0, "feature", name); df.insert(1, "standardize", standardize); df.insert(2, "cv_scheme", cv_name)
            detail.append(df)
            rows.append({"feature": name, "dim": X.shape[1], "standardize": standardize, "cv": cv_name,
                         **summarize_truth(df), "convergence_warnings": n_conv})

interaction_detail = pd.concat(detail, ignore_index=True)
interaction_summary = pd.DataFrame(rows)
interaction_detail.to_csv(OUT_DIR / "interaction_detailed.csv", index=False)
interaction_summary.to_csv(OUT_DIR / "interaction_summary.csv", index=False)
print(interaction_summary.round(4).to_string(index=False))

cosine feature: mean 0.2709, std 0.0346, min 0.1384, max 0.3731
product feature: per-dimension std, mean over dims 0.00085


feature  dim  standardize                            cv  mean_f1  std across relations  median_f1  min_f1  max_f1  mean fold std  baseline_most_frequent_mean_f1  baseline_stratified_mean_f1  n_above_stratified_baseline  n_tied_most_frequent_baseline  n_relations  convergence_warnings
 cosine    1        False                    stratified   0.3713                0.0470     0.3534  0.3377  0.5327         0.0186                          0.3499                       0.4943                            2                             26           36                     0
 cosine    1        False grouped (caption+image joint)   0.3661                0.0377     0.3523  0.3342  0.5093         0.0193                          0.3497                       0.4837                            0                             26           36                     0
 cosine    1         True                    stratified   0.4984                0.0955     0.5105  0.3198  0.6541         0.0717                 

In [4]:
# Per-relation macro F1, one column per (feature, standardize), for each CV scheme
for cv_name in interaction_detail["cv_scheme"].unique():
    sub = interaction_detail[interaction_detail["cv_scheme"] == cv_name]
    wide = sub.pivot_table(index=["relation", "n_samples"], columns=["feature", "standardize"], values="f1_macro_mean")
    base = sub[(sub.feature == "concat") & (~sub.standardize)].set_index(["relation", "n_samples"])
    wide[("baseline", "stratified")] = base["baseline_stratified_f1_macro"]
    wide[("baseline", "most_frequent")] = base["baseline_most_frequent_f1_macro"]
    wide = wide.sort_index(level="n_samples", ascending=False)
    print(f"\nCV: {cv_name}")
    print(wide.round(3).to_string())


CV: stratified
feature                        concat        cosine        product          baseline              
standardize                     False   True  False   True   False   True stratified most_frequent
relation             n_samples                                                                    
touching             884        0.623  0.616  0.479  0.598   0.343  0.605      0.506         0.343
in front of          528        0.514  0.486  0.513  0.533   0.334  0.513      0.498         0.334
behind               498        0.538  0.500  0.346  0.410   0.346  0.493      0.536         0.346
under                415        0.474  0.519  0.370  0.564   0.344  0.528      0.505         0.344
on                   408        0.600  0.555  0.389  0.654   0.344  0.573      0.517         0.344
on top of            349        0.579  0.582  0.348  0.547   0.342  0.584      0.516         0.342
at the right side of 335        0.492  0.507  0.338  0.409   0.338  0.451      0.538         

## 3. Bag-of-Words Baseline (relation identity)

Same task as `probe_relation_identity`: for each relation with at least 50 examples, a one-vs-rest target over all 7,680 captions, `StratifiedKFold(5, shuffle=True, random_state=42)`. The model is `CountVectorizer()` (lowercased unigram counts, vocabulary fit on each training fold) followed by `LogisticRegression(C=1.0)`. The SBERT and CLIP Text columns are read from `results/relation_identity/probing_results.csv`.

In [5]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, f1_score, accuracy_score

scoring = {
    "accuracy": make_scorer(accuracy_score),
    "f1_macro": make_scorer(f1_score, average="macro", zero_division=0),
}
print("Vocabulary size on all captions:", len(CountVectorizer().fit(captions).vocabulary_))

bow_rows = []
for relation in sorted(np.unique(relation_types)):
    target = (relation_types == relation).astype(int)
    if target.sum() < 50:
        continue
    pipe = make_pipeline(CountVectorizer(),
                         LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs", random_state=42))
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    res = cross_validate(pipe, captions, target, cv=cv, scoring=scoring)
    bow_rows.append({"relation": relation, "n_samples": int(target.sum()),
                     "bow_f1": res["test_f1_macro"].mean(), "bow_f1_std": res["test_f1_macro"].std(),
                     "bow_accuracy": res["test_accuracy"].mean()})

bow = pd.DataFrame(bow_rows)
identity = pd.read_csv(RESULTS_DIR / "relation_identity" / "probing_results.csv")
bow = bow.merge(identity, on="relation", how="left").sort_values(
    ["n_samples", "relation"], ascending=[False, True]).reset_index(drop=True)
bow.to_csv(OUT_DIR / "bow_relation_identity.csv", index=False)
print(bow.round(4).to_string(index=False))

Vocabulary size on all captions: 151


            relation  n_samples  bow_f1  bow_f1_std  bow_accuracy  sbert_f1  clip_text_f1  clip_image_f1  clip_concat_f1
            touching        884     1.0         0.0           1.0    0.9893        0.9815         0.4695          0.9017
         in front of        528     1.0         0.0           1.0    0.9214        0.7335         0.4822          0.4841
              behind        498     1.0         0.0           1.0    0.9722        0.7868         0.4832          0.4913
               under        415     1.0         0.0           1.0    0.8754        0.7803         0.4861          0.5434
                  on        408     1.0         0.0           1.0    0.7723        0.6625         0.4864          0.4864
           on top of        349     1.0         0.0           1.0    0.9515        0.9514         0.4884          0.7244
at the right side of        335     1.0         0.0           1.0    0.9315        0.6328         0.4889          0.4889
 at the left side of        273 

In [6]:
cols = ["bow_f1", "sbert_f1", "clip_text_f1", "clip_image_f1", "clip_concat_f1"]
bow_summary = bow[cols].agg(["mean", "std", "median", "min", "max"]).T
bow_summary = bow_summary.rename(columns={"std": "std across relations"})
bow_summary.loc[:, "std across relations"] = bow[cols].std(ddof=0)
bow_summary.to_csv(OUT_DIR / "bow_relation_identity_summary.csv")
print(bow_summary.round(4).to_string())

print("\nBag-of-words vs each condition (wins / ties / losses for bag-of-words, tie = |diff| < 1e-9):")
for c in cols[1:]:
    d = (bow["bow_f1"] - bow[c]).to_numpy()
    print(f"  vs {c:15s}: {(d >= TOL).sum()} / {(np.abs(d) < TOL).sum()} / {(d <= -TOL).sum()}")

                  mean  std across relations  median     min     max
bow_f1          1.0000                0.0000  1.0000  1.0000  1.0000
sbert_f1        0.7097                0.1856  0.6609  0.4974  0.9893
clip_text_f1    0.5891                0.1375  0.4983  0.4929  0.9815
clip_image_f1   0.4933                0.0059  0.4950  0.4695  0.4983
clip_concat_f1  0.5138                0.0760  0.4962  0.4841  0.9017

Bag-of-words vs each condition (wins / ties / losses for bag-of-words, tie = |diff| < 1e-9):
  vs sbert_f1       : 36 / 0 / 0
  vs clip_text_f1   : 36 / 0 / 0
  vs clip_image_f1  : 36 / 0 / 0
  vs clip_concat_f1 : 36 / 0 / 0


## 4. Regularization Sweep and Scaling (truth probe)

`probe_truth_by_relation` for every combination of: four conditions × `C` in `[0.01, 0.1, 1, 10, 100, 1000]` × `StandardScaler` off/on × stratified/grouped folds. `C=1.0` without scaling is the setting used in notebook 03. All settings are reported; none is selected.

In [7]:
C_GRID = [0.01, 0.1, 1, 10, 100, 1000]

detail, rows = [], []
for m in MODELS:
    for cv_name, groups in [("stratified", None), ("grouped", GROUPS[m])]:
        for standardize in [False, True]:
            for C in C_GRID:
                df, n_conv = run_truth(embeddings[m], groups=groups, C=C, standardize=standardize)
                df.insert(0, "condition", m); df.insert(1, "cv_scheme", cv_name)
                df.insert(2, "C", C); df.insert(3, "standardize", standardize)
                detail.append(df)
                rows.append({"condition": m, "cv": cv_name, "standardize": standardize, "C": C,
                             **summarize_truth(df), "convergence_warnings": n_conv})
    print(f"done: {m}")

sweep_detail = pd.concat(detail, ignore_index=True)
sweep_summary = pd.DataFrame(rows)
sweep_detail.to_csv(OUT_DIR / "truth_probe_C_sweep_detailed.csv", index=False)
sweep_summary.to_csv(OUT_DIR / "truth_probe_C_sweep_summary.csv", index=False)
print(f"{len(sweep_summary)} settings, {len(sweep_detail)} per-relation rows")
print("Total convergence warnings:", int(sweep_summary["convergence_warnings"].sum()))

done: sbert


done: clip_text


done: clip_image


done: clip_concat
96 settings, 3456 per-relation rows
Total convergence warnings: 0


In [8]:
def sweep_pivot(value):
    p = sweep_summary.pivot_table(index=["cv", "condition", "standardize"], columns="C", values=value)
    return p.reindex(MODELS, level="condition")

print("Mean macro F1 across 36 relations")
print(sweep_pivot("mean_f1").round(4).to_string())
print("\nMedian macro F1 across 36 relations")
print(sweep_pivot("median_f1").round(4).to_string())
print("\nNumber of relations (of 36) above the stratified-dummy baseline")
print(sweep_pivot("n_above_stratified_baseline").astype(int).to_string())
print("\nNumber of relations (of 36) exactly tied with the most_frequent baseline")
print(sweep_pivot("n_tied_most_frequent_baseline").astype(int).to_string())
print("\nBaselines (mean macro F1 across relations), by cv and condition")
print(sweep_summary.groupby(["cv", "condition"])[["baseline_most_frequent_mean_f1", "baseline_stratified_mean_f1"]]
      .first().reindex(MODELS, level="condition").round(4).to_string())

Mean macro F1 across 36 relations
C                                   0.01     0.10     1.00     10.00    100.00   1000.00
cv         condition   standardize                                                      
grouped    sbert       False         0.3573   0.3918   0.4635   0.4855   0.4889   0.4871
                       True          0.4897   0.4906   0.4870   0.4861   0.4855   0.4851
           clip_text   False         0.3559   0.3805   0.4449   0.4803   0.4865   0.4846
                       True          0.4793   0.4869   0.4889   0.4873   0.4852   0.4819
           clip_image  False         0.3584   0.4034   0.5026   0.5462   0.5447   0.5396
                       True          0.5421   0.5477   0.5426   0.5426   0.5437   0.5425
           clip_concat False         0.3559   0.3828   0.4876   0.5593   0.5653   0.5628
                       True          0.5416   0.5543   0.5527   0.5515   0.5573   0.5572
stratified sbert       False         0.3527   0.3734   0.3696   0.3317   0.3

In [9]:
print("Files in results/diagnostics/:")
for f in sorted(OUT_DIR.glob("*.csv")):
    print(f"  {f.name}")

Files in results/diagnostics/:
  bow_relation_identity.csv
  bow_relation_identity_summary.csv
  interaction_detailed.csv
  interaction_summary.csv
  truth_probe_C_sweep_detailed.csv
  truth_probe_C_sweep_summary.csv
